# Deepfake Image Detector: GPU Training on Google Colab

**Execution Notice:** This notebook has **NOT been executed** on the local CPU workstation, as it is configured specifically for cloud GPU environments (e.g., Google Colab with an NVIDIA T4 GPU). All CLI commands, environment variables, and module flags are statically validated against the repository code.

This workflow provides the route to scale training across larger sample sizes with CUDA acceleration:

* **Option D (Recommended):** Keep the DINOv3 ConvNeXt-Tiny backbone frozen and fit the calibrated linear classification head on a larger training sample. This reproduces the architecture of the default app model (`convnext_tiny_dinov3_linear_v1`).
* **Option B (Experimental):** Full end-to-end fine-tuning of the ConvNeXt-Tiny network using fp16 mixed precision.

Before running: select **Runtime > Change runtime type > T4 GPU**.

Dataset: GenImage (licensed CC BY-NC-SA 4.0 for non-commercial research/educational use). Model weights are initialized from Meta's DINOv3 (DINOv3 License; see `NOTICE.md`).

In [ ]:
!nvidia-smi

## 1. Get the project onto Colab

Zip the project folder on your computer (leave out `data/`, `app_data/`, `artifacts/`, `logs/` and `.env`), upload the zip to
your Google Drive as `Deepfake-Image-Detector.zip`, then run the next cell.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/Deepfake-Image-Detector
!unzip -q /content/drive/MyDrive/Deepfake-Image-Detector.zip -d /content
%cd /content/Deepfake-Image-Detector
!ls

## 2. Install Dependencies

Colab ships PyTorch with CUDA pre-installed, so we do **not** reinstall torch. `timm` is pinned to the project version.

In [ ]:
!pip install -q timm==1.0.30 pyarrow psutil pyyaml scikit-learn pandas matplotlib requests scipy

## 3. Point the project at Colab's disk and the bigger data recipe

`/content` is wiped when the session ends. Save finished artifacts to Drive (last step) or you will lose them.

In [ ]:
import os
os.environ['DFD_DATA_DIR'] = '/content/data'
os.environ['DFD_ARTIFACTS_DIR'] = '/content/artifacts'
os.environ['DFD_DATA_CONFIG'] = 'configs/data_colab.yaml'

## 4. Data: download, validate, split, audit

Each step is resumable. If Colab disconnects, reconnect, re-run steps 1-3 and then this cell again.
The plan (from a dry run of `configs/data_colab.yaml` on the laptop) is about 75,000 images, roughly 15.6 GB of downloads.
The time this takes on Colab is an estimate, not measured: allow one to two hours and expect to re-run the cell after a disconnect.
BigGAN's official train folder only holds about 3,200 images per class, so you will see one "could not reach the requested counts"
warning for it; that is expected.

In [ ]:
!python -m src.data.acquire --workers 16
!python -m src.data.acquire --rebuild-manifest
!python -m src.data.prepare --workers 4
!python -m src.data.splits
!python -m src.data.audit

## 5. Choose the model id for this run

Everything after this cell uses `MODEL_ID`. Use a **new** id (not `convnext_tiny_dinov3_linear_v1`) so that you never
overwrite the laptop's results. Results from this data recipe use different test images than the laptop run and are **not**
directly comparable with the laptop numbers.

In [ ]:
MODEL_ID = 'convnext_tiny_dinov3_linear_colab'   # option D; for option B use e.g. 'convnext_tiny_dinov3_colab'

## 6a. Option D: frozen backbone + linear head (recommended)

Extracts frozen features for the training sample and for every evaluation split, picks the regularisation strength on the
*validation* split only, and folds the fitted head into the network's last layer so the result is an ordinary artifact.
It also writes the predictions for validation / calibration / test / unseen images, so no separate scoring step is needed.
`--train-per-cell 2000` (2,000 images per generator and class) is a larger sample than the laptop's 500; adjust to taste.

In [ ]:
!python -m src.training.linear_head --model convnext_tiny_dinov3 --model-id {MODEL_ID} --device cuda --batch-size 128 --train-per-cell 2000

## 6b. Option B (optional): fine-tune the DINOv3 ConvNeXt end to end

Skip this cell if you ran 6a. fp16 mixed precision; the learning rates are *untested guesses* (the laptop used 5e-5 / 5e-4 for
1,000 steps and ended up over-fitted to its six training generators). `--resume` continues an interrupted run.
Remember to change `MODEL_ID` above first.

In [ ]:
!python -m src.training.train --model convnext_tiny_dinov3 --model-id {MODEL_ID} --device cuda --amp --workers 2 \
    --batch-size 64 --epochs 8 --samples-per-epoch 12000 --eval-every 50 --lr 3e-5 --head-lr 3e-4

## 7. Evaluate and Calibrate Model

Option B only: score the evaluation splits first (option D already wrote its predictions in step 6a, so skip this cell).

In [ ]:
!python -m src.evaluation.run_eval --model-id {MODEL_ID} --splits validation calibration test unseen_test --device cuda

Both options: calibrate, measure speed and robustness, and write the report.

In [ ]:
!python -m src.evaluation.calibrate --model-id {MODEL_ID}
!python -m src.evaluation.run_eval --model-id {MODEL_ID} --latency --device cuda
!python -m src.evaluation.robustness --model-id {MODEL_ID} --device cuda
!python -m src.evaluation.report --model-id {MODEL_ID}

## 8. Export Results

Copy the model artifacts, calibration parameters, and evaluation reports to Google Drive, then download the zip package.

In [ ]:
!mkdir -p /content/results
!cp -r /content/artifacts/{MODEL_ID} /content/results/artifacts_{MODEL_ID}
!rm -f /content/results/artifacts_{MODEL_ID}/last.pt /content/results/artifacts_{MODEL_ID}/best_state.pt
!cp -r reports/{MODEL_ID} /content/results/reports_{MODEL_ID}
!cp reports/predictions/{MODEL_ID}.csv /content/results/
!cd /content && zip -qr colab_results.zip results
!cp /content/colab_results.zip /content/drive/MyDrive/
from google.colab import files
files.download('/content/colab_results.zip')

## 9. Deploy Trained Artifact to the Application

On your computer, extract `colab_results.zip` (replacing `<MODEL_ID>` with your identifier):

1. Copy `artifacts_<MODEL_ID>/` to `Deepfake-Image-Detector/artifacts/<MODEL_ID>/`
2. Copy `reports_<MODEL_ID>/` to `reports/<MODEL_ID>/` and the CSV to `reports/predictions/`
3. To activate the new model in the app, write the folder name into `artifacts/DEFAULT_MODEL`
4. Start the application: `streamlit run app.py`

Note: Metrics from different sampling recipes are evaluated on different test sets. Keep record of the training configuration and hardware alongside reported scores.